# SPRINT 1

In [1]:
import pandas as pd

# Ler o arquivo CSV
df = pd.read_csv("halloween_vendas.csv", sep=",", encoding="utf-8")

# Conferir total de linhas
total_linhas = len(df)
print(f"O arquivo foi lido com sucesso! Total de linhas encontradas: {total_linhas}")

# Mostrar 3 registros
df.sample(3)


O arquivo foi lido com sucesso! Total de linhas encontradas: 1500


,pedido_id,data_venda,produto,categoria,canal_venda,cidade,quantidade,preco_unitario,custo_unitario,desconto_pct,status_pedido,publico,tamanho,data_entrega
1279,83,2024-10-25 14:16:40,Fantasia de bruxa,Fantasia,Loja física,Cascavel,2,110.0,57.0,5,Concluído,Adulto,M,2024-10-30 14:16:40
1012,402,2024-10-31 12:00:33,Fantasia de abobora,Fantasia,Mercado,Maringá,6,85.0,44.0,0,Concluído,Infantil,P,2024-11-04 12:00:33
430,652,2024-09-16 19:27:23,Chapeu de bruxa,Acessorio,Loja física,Cascavel,2,22.0,8.0,0,Concluído,Todos,Unico,2024-09-20 19:27:23


# SPRINT 2 - Limpar e padronizar

In [2]:
import re

#  Vazio ou nulo (NaN) vira "sem categoria"
def limpar_categoria(texto):
    if pd.isna(texto) or str(texto).strip() == "":
        return "sem categoria"

    # Transforma dado em texto
    texto_str = str(texto)
        
    # Remove espaços em branco
    texto_sem_espacos = texto_str.strip()
        
    # Transforma todas as letras em minúsculas
    texto_minusculo = texto_sem_espacos.lower()
        
    # Tira pontuações
    texto_limpo = re.sub(r'[^a-z\s]', '', texto_minusculo)

    if texto_limpo.strip() == "":
        return "sem categoria"
        
    return texto_limpo

def padronizar_canal(texto):
    if pd.isna(texto):
        return "Não reconhecido"

    # Transforma dado em texto
    canal_str = str(texto)
    
    # Remove espaços em branco
    canal_sem_espacos = canal_str.strip()
    
    # Transforma todas as letras em minúsculas
    canal_minusculo = canal_sem_espacos.lower()

    #Padronizando tudo
       # #Padronizando tudo
    if "mercado" in canal_minusculo:
        return "Mercado"
    elif "loja" in canal_minusculo or "fisica" in canal_minusculo or "física" in canal_minusculo:
        return "Loja física"
    elif "web" in canal_minusculo:
        return "Web"
    else:
        return "Não reconhecido"

# Contagem:

# Cópia da coluna original para conseguir comparar o antes/depois
categorias_originais = df['categoria'].copy()

# Novas com os dados limpos para não estragar os originais
df['categoria_limpa'] = df['categoria'].apply(limpar_categoria)
df['canal_venda_limpo'] = df['canal_venda'].apply(padronizar_canal)

# Categorias vazias preenchidas
vazias_preenchidas = ((categorias_originais.isna()) | (categorias_originais.astype(str).str.strip() == "")).sum()

# Categorias não vazias alteradas
alteradas = 0
for orig, limpo in zip(categorias_originais, df['categoria_limpa']):
    if pd.notna(orig) and str(orig).strip() != "":
        if str(orig) != limpo:
            alteradas += 1

# Resultados:

print("--- RESULTADOS SPRINT 2 ---")
print(f"Categorias vazias preenchidas: {vazias_preenchidas}")
print(f"Categorias não vazias alteradas: {alteradas}")
print("\nExemplo de antes e depois dos dados na tabela:")
df[['categoria', 'categoria_limpa', 'canal_venda', 'canal_venda_limpo']].head(10)


--- RESULTADOS SPRINT 2 ---
Categorias vazias preenchidas: 18
Categorias não vazias alteradas: 1482

Exemplo de antes e depois dos dados na tabela:


,categoria,categoria_limpa,canal_venda,canal_venda_limpo
0,Fantasia,fantasia,Loja física,Loja física
1,FANTASIA!!!,fantasia,Loja física,Loja física
2,Fantasia,fantasia,Web,Web
3,Acessorio,acessorio,Loja física,Loja física
4,Acessorio,acessorio,Mercado,Mercado
5,Fantasia,fantasia,Loja física,Loja física
6,Fantasia,fantasia,Web,Web
7,Acessorio,acessorio,Loja física,Loja física
8,Fantasia,fantasia,Web,Web
9,Fantasia,fantasia,Loja física,Loja física


# SPRINT 3

In [3]:
# Regra: quantidade válida = maior que zero; preço válido = preenchido e não negativo.
# Registros fora da regra só são MARCADOS, nunca corrigidos, para não inventar valores.
# Pedido com os dois erros conta uma vez no total.

#Identificar problemas na coluna Quantidade (vazia/NaN ou menor/igual a zero)
erros_quantidade = (df['quantidade'].isna()) | (df['quantidade'] <= 0)

#Identificar os problemas na coluna Preço Unitário (vazia/NaN ou menor que zero)
erros_preco = (df['preco_unitario'].isna()) | (df['preco_unitario'] < 0)

#Problemas individuais de cada campo 
total_erros_qtd = erros_quantidade.sum()
total_erros_preco = erros_preco.sum()

#Total de pedidos afetados sem sobreposição
pedidos_afetados_total = (erros_quantidade | erros_preco).sum()

#Nova coluna para marcar se o registro tem algum problema numérico
df['com_problema_numerico'] = erros_quantidade | erros_preco

print("--- RESULTADOS SPRINT 3 ---")
print(f"Pedidos com problemas na Quantidade: {total_erros_qtd}")
print(f"Pedidos com problemas no Preço Unitário: {total_erros_preco}")
print(f"Total de pedidos afetados (sem sobreposição): {pedidos_afetados_total}")

--- RESULTADOS SPRINT 3 ---
Pedidos com problemas na Quantidade: 7
Pedidos com problemas no Preço Unitário: 12
Total de pedidos afetados (sem sobreposição): 19


# SPRINT 4

In [4]:
from datetime import datetime

def converter_data(texto):
    if pd.isna(texto) or str(texto).strip() == "":
        return None
    try:
        return datetime.strptime(str(texto).strip(), "%Y-%m-%d %H:%M:%S")
    except:
        return None

#Transforma o texto bruto em formato de data real
df['data_venda_objeto'] = df['data_venda'].apply(converter_data)

#Extrair de ano, mês, dia e hora direto do objeto de data
df['ano'] = df['data_venda_objeto'].apply(lambda x: x.year if x else None)
df['mes'] = df['data_venda_objeto'].apply(lambda x: x.month if x else None)
df['dia'] = df['data_venda_objeto'].apply(lambda x: x.day if x else None)
df['hora'] = df['data_venda_objeto'].apply(lambda x: x.hour if x else None)

#Converter para o formato brasileiro
df['data_venda_ptbr'] = df['data_venda_objeto'].apply(lambda x: x.strftime("%d/%m/%Y") if pd.notnull(x) else "Data Inválida")

#Identificação, contagem e marcação de erros
erros_data = df['data_venda_objeto'].isna()
total_erros_data = erros_data.sum()
df['com_data_invalida'] = erros_data

print("--- RESULTADOS FORMATAÇÃO TEMPORAL ---")
print(f"Total de datas inválidas ou vazias encontradas: {total_erros_data}")
print("\nExemplo das novas colunas de tempo criadas:")

print(df[['data_venda', 'data_venda_ptbr', 'ano', 'mes', 'dia', 'hora']].head(3))
#Pedidos sem data de entrega
sem_entrega = df['data_entrega'].isna()
cancelado = df['status_pedido'] == 'Cancelado'

#Contagem dos dois grupos, separados
total_sem_entrega_cancelado = (sem_entrega & cancelado).sum()
total_outros_status = (sem_entrega & ~cancelado).sum()

print("--- [ENTREGA PARCIAL SPRINT 4: HIPÓTESE] ---")
print(f"Pedidos sem entrega e cancelados: {total_sem_entrega_cancelado}")
print(f"Pedidos sem entrega e com outros status: {total_outros_status}")

if total_outros_status > 0:
    print("RESPOSTA: A hipótese é FALSA nesta base, existem pedidos sem entrega que não estão cancelados.")
    print("\nExemplos (até 3):")
    print(df[sem_entrega & ~cancelado][['pedido_id', 'status_pedido', 'data_venda']].head(3))
else:
    print("RESPOSTA: A hipótese é VERDADEIRA. Todos os pedidos sem entrega constam como Cancelados.")

print("\n" + "="*50 + "\n")

#Pedidos por canal (só canais reconhecidos, todos os status)
canal_reconhecido = df['canal_venda_limpo'] != 'Não reconhecido'
pedidos_por_canal = df[canal_reconhecido]['canal_venda_limpo'].value_counts()
excluidos_canal = (~canal_reconhecido).sum()

#Pedidos por ano-mês (só datas válidas, todos os status)
df['ano_mes'] = df['data_venda_objeto'].apply(lambda x: x.strftime("%Y-%m") if pd.notnull(x) else None)
pedidos_por_ano_mes = df[~df['com_data_invalida']].groupby('ano_mes').size()

print("--- [ENTREGA PARCIAL SPRINT 4: DESCOBERTAS] ---")

print("\nDESCOBERTA 1 (Canal):")
print(pedidos_por_canal.to_string())
print(f"-> O canal com mais pedidos é {pedidos_por_canal.idxmax()}, com {pedidos_por_canal.max()} de {pedidos_por_canal.sum()} pedidos.")
print(f"-> Pedidos excluídos por canal não reconhecido: {excluidos_canal}")
print("-> Limitação: são pedidos registrados (inclui cancelados e devolvidos), não receita nem vendas concluídas.")

print("\nDESCOBERTA 2 (Período):")
print(pedidos_por_ano_mes.to_string())
print(f"-> O período com mais pedidos é {pedidos_por_ano_mes.idxmax()}, com {pedidos_por_ano_mes.max()} pedidos.")
print(f"-> Pedidos excluídos por data inválida: {total_erros_data}")
print("-> Limitação: a base cobre só agosto a novembro de cada ano e é fictícia.")

--- RESULTADOS FORMATAÇÃO TEMPORAL ---
Total de datas inválidas ou vazias encontradas: 4

Exemplo das novas colunas de tempo criadas:
            data_venda data_venda_ptbr     ano   mes   dia  hora
0  2025-11-21 18:02:35      21/11/2025  2025.0  11.0  21.0  18.0
1  2024-10-29 18:41:26      29/10/2024  2024.0  10.0  29.0  18.0
2  2025-09-05 21:16:09      05/09/2025  2025.0   9.0   5.0  21.0
--- [ENTREGA PARCIAL SPRINT 4: HIPÓTESE] ---
Pedidos sem entrega e cancelados: 109
Pedidos sem entrega e com outros status: 34
RESPOSTA: A hipótese é FALSA nesta base, existem pedidos sem entrega que não estão cancelados.

Exemplos (até 3):
     pedido_id status_pedido           data_venda
49        1170     Concluído  2025-08-19 22:20:19
117        990     Concluído  2025-10-13 20:00:16
152        740     Concluído        data_invalida


--- [ENTREGA PARCIAL SPRINT 4: DESCOBERTAS] ---

DESCOBERTA 1 (Canal):
canal_venda_limpo
Web            666
Loja física    551
Mercado        283
-> O canal com ma

SPRINT 5

In [5]:
total_cancelados = (df['status_pedido'] == 'Cancelado').sum()

print("=" * 50)
print("RELATÓRIO DE STATUS - FESTA EM CENA")
print("=" * 50)
print(f"Total de linhas lidas: {total_linhas}")
print(f"Categorias vazias preenchidas: {vazias_preenchidas}")
print(f"Categorias não vazias alteradas: {alteradas}")
print(f"Quantidade inválida: {total_erros_qtd}")
print(f"Preço inválido ou ausente: {total_erros_preco}")
print(f"Problemas numéricos (pedidos afetados): {pedidos_afetados_total}")
print(f"Datas inválidas: {total_erros_data}")
print(f"Total de cancelados: {total_cancelados}")
print(f"Sem entrega e cancelados: {total_sem_entrega_cancelado}")
print(f"Sem entrega e outros status: {total_outros_status}")

RELATÓRIO DE STATUS - FESTA EM CENA
Total de linhas lidas: 1500
Categorias vazias preenchidas: 18
Categorias não vazias alteradas: 1482
Quantidade inválida: 7
Preço inválido ou ausente: 12
Problemas numéricos (pedidos afetados): 19
Datas inválidas: 4
Total de cancelados: 109
Sem entrega e cancelados: 109
Sem entrega e outros status: 34
